In [ ]:
# ============================================================
# NB4 — EYE DETECTION & DROWSINESS ANALYSIS
# BLOCK 1 — IMPORT LIBRARIES
# ============================================================

import os
from pathlib import Path
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt

import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("=" * 60)
print("NB4 — Eye Detection")
print("=" * 60)

print("OpenCV      :", cv2.__version__)
print("NumPy       :", np.__version__)
print("MediaPipe   :", mp.__version__)

print("=" * 60)

NB4 — Eye Detection
OpenCV      : 5.0.0
NumPy       : 2.2.6
MediaPipe   : 1.0.0


In [ ]:
# ============================================================
# NB4 — BLOCK 2
# LOAD MEDIAPIPE FACE LANDMARKER
# ============================================================

MODEL_PATH = Path.cwd() / "assets" / "face_landmarker.task"

# ------------------------------------------------------------
# Verify model exists
# ------------------------------------------------------------

if not os.path.exists(MODEL_PATH):

    raise FileNotFoundError(
        f"MediaPipe model not found:\n{MODEL_PATH}"
    )


# ------------------------------------------------------------
# MediaPipe configuration
# ------------------------------------------------------------

BaseOptions = python.BaseOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
RunningMode = vision.RunningMode


options = FaceLandmarkerOptions(

    base_options=BaseOptions(
        model_asset_path=str(MODEL_PATH)
    ),

    running_mode=RunningMode.IMAGE,

    num_faces=1
)


# ------------------------------------------------------------
# Create detector
# ------------------------------------------------------------

detector = FaceLandmarker.create_from_options(
    options
)


print("=" * 60)
print("MEDIAPIPE FACE LANDMARKER LOADED")
print("=" * 60)

print("Model :", MODEL_PATH)
print("Mode  : IMAGE")
print("Faces : 1")

print("=" * 60)

MEDIAPIPE FACE LANDMARKER LOADED
Model : assets/face_landmarker.task
Mode  : IMAGE
Faces : 1


In [3]:
# ============================================================
# NB4 — BLOCK 3
# EYE LANDMARK INDICES
# ============================================================

# ------------------------------------------------------------
# Six landmarks around the LEFT eye
# ------------------------------------------------------------

LEFT_EYE = [
    33,    # p1 - outer corner
    160,   # p2 - upper
    158,   # p3 - upper
    133,   # p4 - inner corner
    153,   # p5 - lower
    144    # p6 - lower
]


# ------------------------------------------------------------
# Six landmarks around the RIGHT eye
# ------------------------------------------------------------

RIGHT_EYE = [
    362,   # p1 - inner corner
    385,   # p2 - upper
    387,   # p3 - upper
    263,   # p4 - outer corner
    373,   # p5 - lower
    380    # p6 - lower
]


print("=" * 60)
print("EYE LANDMARKS CONFIGURED")
print("=" * 60)

print("Left Eye  :", LEFT_EYE)
print("Right Eye :", RIGHT_EYE)

print("=" * 60)

EYE LANDMARKS CONFIGURED
Left Eye  : [33, 160, 158, 133, 153, 144]
Right Eye : [362, 385, 387, 263, 373, 380]


In [5]:
# ============================================================
# NB4 — BLOCK 4
# LIVE EYE LANDMARK VISUALIZATION
# ============================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("Eye landmark visualization started.")
print("Press Q to quit.")


while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to read webcam frame.")
        break

    # --------------------------------------------------------
    # Convert frame for MediaPipe
    # --------------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # --------------------------------------------------------
    # Detect face landmarks
    # --------------------------------------------------------

    result = detector.detect(mp_image)

    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        h, w = frame.shape[:2]

        # ----------------------------------------------------
        # Draw LEFT eye landmarks
        # ----------------------------------------------------

        for idx in LEFT_EYE:

            x = int(landmarks[idx].x * w)
            y = int(landmarks[idx].y * h)

            cv2.circle(
                frame,
                (x, y),
                4,
                (0, 255, 0),
                -1
            )

        # ----------------------------------------------------
        # Draw RIGHT eye landmarks
        # ----------------------------------------------------

        for idx in RIGHT_EYE:

            x = int(landmarks[idx].x * w)
            y = int(landmarks[idx].y * h)

            cv2.circle(
                frame,
                (x, y),
                4,
                (0, 255, 0),
                -1
            )

        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        cv2.putText(
            frame,
            "Eye landmarks detected",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

    else:

        cv2.putText(
            frame,
            "Face not detected",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    cv2.imshow(
        "NB4 - Eye Landmarks",
        frame
    )

    # --------------------------------------------------------
    # Quit
    # --------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

print("Eye landmark test stopped.")

Eye landmark visualization started.
Press Q to quit.
Eye landmark test stopped.


In [6]:
# ============================================================
# NB4 — BLOCK 5
# EYE ASPECT RATIO (EAR)
# ============================================================

def calculate_ear(landmarks, eye_indices, frame_width, frame_height):
    """
    Calculate Eye Aspect Ratio (EAR)
    from six eye landmarks.

    eye_indices:
        [p1, p2, p3, p4, p5, p6]
    """

    # --------------------------------------------------------
    # Convert normalized MediaPipe coordinates
    # to pixel coordinates
    # --------------------------------------------------------

    points = []

    for idx in eye_indices:

        x = landmarks[idx].x * frame_width
        y = landmarks[idx].y * frame_height

        points.append(
            np.array([x, y])
        )

    p1, p2, p3, p4, p5, p6 = points

    # --------------------------------------------------------
    # Vertical eye distances
    # --------------------------------------------------------

    vertical_1 = np.linalg.norm(
        p2 - p6
    )

    vertical_2 = np.linalg.norm(
        p3 - p5
    )

    # --------------------------------------------------------
    # Horizontal eye distance
    # --------------------------------------------------------

    horizontal = np.linalg.norm(
        p1 - p4
    )

    # --------------------------------------------------------
    # EAR
    # --------------------------------------------------------

    if horizontal == 0:
        return 0.0

    ear = (
        vertical_1 + vertical_2
    ) / (
        2.0 * horizontal
    )

    return float(ear)


print("=" * 60)
print("EAR FUNCTION READY")
print("=" * 60)

print("Formula:")
print("EAR = (vertical_1 + vertical_2) / (2 × horizontal)")

print("=" * 60)

EAR FUNCTION READY
Formula:
EAR = (vertical_1 + vertical_2) / (2 × horizontal)


In [8]:
# ============================================================
# NB4 — BLOCK 6
# LIVE EAR MONITOR
# ============================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("=" * 60)
print("LIVE EAR MONITOR")
print("=" * 60)
print("Look normally with your eyes open.")
print("Blink naturally.")
print("Then close both eyes for a few seconds.")
print("Press Q to quit.")
print("=" * 60)


while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to read webcam frame.")
        break

    # --------------------------------------------------------
    # Frame dimensions
    # --------------------------------------------------------

    h, w = frame.shape[:2]

    # --------------------------------------------------------
    # Convert BGR → RGB
    # --------------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # --------------------------------------------------------
    # Detect face landmarks
    # --------------------------------------------------------

    result = detector.detect(mp_image)


    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        # ----------------------------------------------------
        # Calculate left EAR
        # ----------------------------------------------------

        left_ear = calculate_ear(
            landmarks,
            LEFT_EYE,
            w,
            h
        )

        # ----------------------------------------------------
        # Calculate right EAR
        # ----------------------------------------------------

        right_ear = calculate_ear(
            landmarks,
            RIGHT_EYE,
            w,
            h
        )

        # ----------------------------------------------------
        # Average EAR
        # ----------------------------------------------------

        average_ear = (
            left_ear + right_ear
        ) / 2.0


        # ----------------------------------------------------
        # Display values
        # ----------------------------------------------------

        cv2.putText(
            frame,
            f"Left EAR  : {left_ear:.3f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Right EAR : {right_ear:.3f}",
            (20, 75),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Average EAR: {average_ear:.3f}",
            (20, 110),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 0),
            2
        )


    else:

        cv2.putText(
            frame,
            "Face not detected",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )


    # --------------------------------------------------------
    # Display camera
    # --------------------------------------------------------

    cv2.imshow(
        "NB4 - Live EAR Monitor",
        frame
    )


    # --------------------------------------------------------
    # Quit
    # --------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

print("EAR monitor stopped.")

LIVE EAR MONITOR
Look normally with your eyes open.
Blink naturally.
Then close both eyes for a few seconds.
Press Q to quit.
EAR monitor stopped.


In [9]:
# ============================================================
# NB4 — BLOCK 7
# EYE STATE CLASSIFICATION
# ============================================================

# Initial threshold based on your measured values
EAR_THRESHOLD = 0.19


def classify_eye_state(ear):
    """
    Classify eye as OPEN or CLOSED
    using the calibrated EAR threshold.
    """

    if ear >= EAR_THRESHOLD:
        return "OPEN"

    return "CLOSED"


print("=" * 60)
print("EYE STATE CLASSIFIER READY")
print("=" * 60)

print(f"EAR Threshold : {EAR_THRESHOLD:.3f}")

print()
print("EAR >= threshold → OPEN")
print("EAR <  threshold → CLOSED")

print("=" * 60)

EYE STATE CLASSIFIER READY
EAR Threshold : 0.190

EAR >= threshold → OPEN
EAR <  threshold → CLOSED


In [11]:
# ============================================================
# NB4 — BLOCK 8
# LIVE EYE STATE DETECTION — MIRRORED CAMERA
# ============================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("=" * 60)
print("LIVE EYE STATE DETECTION")
print("=" * 60)
print("Camera is mirrored.")
print("Blink normally.")
print("Close both eyes for a few seconds.")
print("Press Q to quit.")
print("=" * 60)


while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to read webcam frame.")
        break

    # --------------------------------------------------------
    # MIRROR CAMERA
    # --------------------------------------------------------

    frame = cv2.flip(frame, 1)

    # --------------------------------------------------------
    # Frame dimensions
    # --------------------------------------------------------

    h, w = frame.shape[:2]

    # --------------------------------------------------------
    # Convert BGR → RGB
    # --------------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # --------------------------------------------------------
    # Detect face landmarks
    # --------------------------------------------------------

    result = detector.detect(mp_image)

    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        # ----------------------------------------------------
        # Calculate EAR
        # ----------------------------------------------------

        left_ear = calculate_ear(
            landmarks,
            LEFT_EYE,
            w,
            h
        )

        right_ear = calculate_ear(
            landmarks,
            RIGHT_EYE,
            w,
            h
        )

        average_ear = (
            left_ear + right_ear
        ) / 2.0

        # ----------------------------------------------------
        # Classify each eye
        # ----------------------------------------------------

        left_state = classify_eye_state(
            left_ear
        )

        right_state = classify_eye_state(
            right_ear
        )

        # ----------------------------------------------------
        # Combined eye state
        # ----------------------------------------------------

        if (
            left_state == "CLOSED"
            and
            right_state == "CLOSED"
        ):

            combined_state = "BOTH EYES CLOSED"

        elif (
            left_state == "OPEN"
            and
            right_state == "OPEN"
        ):

            combined_state = "BOTH EYES OPEN"

        else:

            combined_state = "ONE EYE CLOSED"

        # ----------------------------------------------------
        # Display EAR values
        # ----------------------------------------------------

        cv2.putText(
            frame,
            f"Left EAR  : {left_ear:.3f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Right EAR : {right_ear:.3f}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Average EAR : {average_ear:.3f}",
            (20, 105),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 0),
            2
        )

        # ----------------------------------------------------
        # Display eye states
        # ----------------------------------------------------

        cv2.putText(
            frame,
            f"Left Eye  : {left_state}",
            (20, 145),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Right Eye : {right_state}",
            (20, 180),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 0),
            2
        )

        # ----------------------------------------------------
        # Combined state
        # ----------------------------------------------------

        cv2.putText(
            frame,
            combined_state,
            (20, 225),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 255),
            2
        )

    else:

        cv2.putText(
            frame,
            "FACE NOT DETECTED",
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 0, 255),
            2
        )

    # --------------------------------------------------------
    # Show camera
    # --------------------------------------------------------

    cv2.imshow(
        "NB4 - Live Eye State",
        frame
    )

    # --------------------------------------------------------
    # Quit
    # --------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()
cv2.destroyAllWindows()

print("Eye state detection stopped.")

LIVE EYE STATE DETECTION
Camera is mirrored.
Blink normally.
Close both eyes for a few seconds.
Press Q to quit.
Eye state detection stopped.


In [12]:
# ============================================================
# NB4 — BLOCK 9
# EYE CLOSURE DURATION
# ============================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")


# ------------------------------------------------------------
# Closure tracking variables
# ------------------------------------------------------------

eyes_closed = False

closure_start_time = None

closure_duration = 0.0

longest_closure = 0.0


print("=" * 60)
print("EYE CLOSURE DURATION TEST")
print("=" * 60)
print("Blink normally.")
print("Then close both eyes for different durations.")
print("Press Q to quit.")
print("=" * 60)


while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to read webcam frame.")
        break


    # --------------------------------------------------------
    # Mirror camera
    # --------------------------------------------------------

    frame = cv2.flip(frame, 1)


    # --------------------------------------------------------
    # Frame dimensions
    # --------------------------------------------------------

    h, w = frame.shape[:2]


    # --------------------------------------------------------
    # MediaPipe
    # --------------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)


    if result.face_landmarks:

        landmarks = result.face_landmarks[0]


        # ----------------------------------------------------
        # Calculate EAR
        # ----------------------------------------------------

        left_ear = calculate_ear(
            landmarks,
            LEFT_EYE,
            w,
            h
        )

        right_ear = calculate_ear(
            landmarks,
            RIGHT_EYE,
            w,
            h
        )

        average_ear = (
            left_ear + right_ear
        ) / 2.0


        # ----------------------------------------------------
        # Determine eye states
        # ----------------------------------------------------

        left_state = classify_eye_state(
            left_ear
        )

        right_state = classify_eye_state(
            right_ear
        )


        both_closed = (
            left_state == "CLOSED"
            and
            right_state == "CLOSED"
        )


        # ====================================================
        # CLOSURE TIMER
        # ====================================================

        if both_closed:

            # ----------------------------------------------
            # Eyes have just become closed
            # ----------------------------------------------

            if not eyes_closed:

                eyes_closed = True

                closure_start_time = time.time()


            # ----------------------------------------------
            # Calculate current closure duration
            # ----------------------------------------------

            closure_duration = (
                time.time()
                - closure_start_time
            )


        else:

            # ----------------------------------------------
            # Eyes have opened again
            # ----------------------------------------------

            if eyes_closed:

                # Save longest closure
                if closure_duration > longest_closure:

                    longest_closure = closure_duration


            eyes_closed = False

            closure_start_time = None

            closure_duration = 0.0


        # ----------------------------------------------------
        # Current state
        # ----------------------------------------------------

        if both_closed:

            state_text = "BOTH EYES CLOSED"

        else:

            state_text = "EYES OPEN"


        # ====================================================
        # DISPLAY
        # ====================================================

        cv2.putText(
            frame,
            f"Left EAR: {left_ear:.3f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Right EAR: {right_ear:.3f}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Average EAR: {average_ear:.3f}",
            (20, 105),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"State: {state_text}",
            (20, 145),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Closed Duration: {closure_duration:.2f} sec",
            (20, 185),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Longest Closure: {longest_closure:.2f} sec",
            (20, 225),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 255),
            2
        )


    else:

        cv2.putText(
            frame,
            "FACE NOT DETECTED",
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 0, 255),
            2
        )


    # --------------------------------------------------------
    # Show camera
    # --------------------------------------------------------

    cv2.imshow(
        "NB4 - Eye Closure Duration",
        frame
    )


    # --------------------------------------------------------
    # Quit
    # --------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()

cv2.destroyAllWindows()


print("=" * 60)
print("EYE CLOSURE TEST FINISHED")
print("=" * 60)

print(
    f"Longest detected closure: "
    f"{longest_closure:.2f} seconds"
)

print("=" * 60)

EYE CLOSURE DURATION TEST
Blink normally.
Then close both eyes for different durations.
Press Q to quit.
EYE CLOSURE TEST FINISHED
Longest detected closure: 6.21 seconds


In [13]:
# ============================================================
# NB4 — BLOCK 10
# EYE CLOSURE CLASSIFICATION
# ============================================================

BLINK_THRESHOLD = 0.50
LONG_BLINK_THRESHOLD = 1.00


def classify_eye_closure(duration):
    """
    Classify a continuous eye-closure event
    based on its duration.
    """

    if duration < BLINK_THRESHOLD:

        return "NORMAL BLINK"

    elif duration < LONG_BLINK_THRESHOLD:

        return "LONG BLINK"

    else:

        return "PROLONGED CLOSURE"


print("=" * 60)
print("EYE CLOSURE CLASSIFIER READY")
print("=" * 60)

print(
    f"< {BLINK_THRESHOLD:.2f} sec"
    " → NORMAL BLINK"
)

print(
    f"{BLINK_THRESHOLD:.2f}–"
    f"{LONG_BLINK_THRESHOLD:.2f} sec"
    " → LONG BLINK"
)

print(
    f">= {LONG_BLINK_THRESHOLD:.2f} sec"
    " → PROLONGED CLOSURE"
)

print("=" * 60)

EYE CLOSURE CLASSIFIER READY
< 0.50 sec → NORMAL BLINK
0.50–1.00 sec → LONG BLINK
>= 1.00 sec → PROLONGED CLOSURE


In [14]:
# ============================================================
# NB4 — BLOCK 11
# LIVE BLINK / PROLONGED CLOSURE DETECTION
# ============================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")


# ------------------------------------------------------------
# Closure tracking
# ------------------------------------------------------------

eyes_closed = False

closure_start_time = None

closure_duration = 0.0

last_event = "None"

prolonged_closures = 0

longest_closure = 0.0


print("=" * 60)
print("LIVE BLINK / PROLONGED CLOSURE DETECTION")
print("=" * 60)
print("Blink normally.")
print("Try closing your eyes for 1–3 seconds.")
print("Press Q to quit.")
print("=" * 60)


while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to read webcam frame.")
        break


    # --------------------------------------------------------
    # Mirror camera
    # --------------------------------------------------------

    frame = cv2.flip(frame, 1)


    # --------------------------------------------------------
    # Frame dimensions
    # --------------------------------------------------------

    h, w = frame.shape[:2]


    # --------------------------------------------------------
    # MediaPipe
    # --------------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)


    if result.face_landmarks:

        landmarks = result.face_landmarks[0]


        # ----------------------------------------------------
        # Calculate EAR
        # ----------------------------------------------------

        left_ear = calculate_ear(
            landmarks,
            LEFT_EYE,
            w,
            h
        )

        right_ear = calculate_ear(
            landmarks,
            RIGHT_EYE,
            w,
            h
        )

        average_ear = (
            left_ear + right_ear
        ) / 2.0


        # ----------------------------------------------------
        # Determine eye states
        # ----------------------------------------------------

        left_state = classify_eye_state(
            left_ear
        )

        right_state = classify_eye_state(
            right_ear
        )


        both_closed = (
            left_state == "CLOSED"
            and
            right_state == "CLOSED"
        )


        # ====================================================
        # EYES CLOSED
        # ====================================================

        if both_closed:

            # ----------------------------------------------
            # Closure has just started
            # ----------------------------------------------

            if not eyes_closed:

                eyes_closed = True

                closure_start_time = time.time()

                closure_duration = 0.0


            # ----------------------------------------------
            # Update duration
            # ----------------------------------------------

            closure_duration = (
                time.time()
                - closure_start_time
            )


        # ====================================================
        # EYES OPEN
        # ====================================================

        else:

            # ----------------------------------------------
            # Closure event just ended
            # ----------------------------------------------

            if eyes_closed:

                # Classify completed closure
                event = classify_eye_closure(
                    closure_duration
                )

                last_event = event


                # Update longest closure
                if closure_duration > longest_closure:

                    longest_closure = closure_duration


                # Count prolonged closures
                if event == "PROLONGED CLOSURE":

                    prolonged_closures += 1


            # Reset current closure
            eyes_closed = False

            closure_start_time = None

            closure_duration = 0.0


        # ====================================================
        # CURRENT STATE
        # ====================================================

        if both_closed:

            state_text = "BOTH EYES CLOSED"

        else:

            state_text = "EYES OPEN"


        # ====================================================
        # DISPLAY
        # ====================================================

        cv2.putText(
            frame,
            f"Left EAR  : {left_ear:.3f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Right EAR : {right_ear:.3f}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Average EAR: {average_ear:.3f}",
            (20, 105),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"State: {state_text}",
            (20, 145),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Current Closure: {closure_duration:.2f}s",
            (20, 185),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Last Event: {last_event}",
            (20, 225),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Prolonged Closures: {prolonged_closures}",
            (20, 265),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Longest Closure: {longest_closure:.2f}s",
            (20, 305),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )


    else:

        cv2.putText(
            frame,
            "FACE NOT DETECTED",
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 0, 255),
            2
        )


    # --------------------------------------------------------
    # Display camera
    # --------------------------------------------------------

    cv2.imshow(
        "NB4 - Blink & Closure Detection",
        frame
    )


    # --------------------------------------------------------
    # Quit
    # --------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()

cv2.destroyAllWindows()


print("=" * 60)
print("TEST FINISHED")
print("=" * 60)

print(
    f"Prolonged closures detected: "
    f"{prolonged_closures}"
)

print(
    f"Longest closure: "
    f"{longest_closure:.2f} seconds"
)

print("=" * 60)

LIVE BLINK / PROLONGED CLOSURE DETECTION
Blink normally.
Try closing your eyes for 1–3 seconds.
Press Q to quit.
TEST FINISHED
Prolonged closures detected: 4
Longest closure: 5.41 seconds


In [15]:
# ============================================================
# NB4 — BLOCK 12
# EYE-BASED DROWSINESS SCORE
# ============================================================

from collections import deque


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

# Keep recent closure events
EVENT_HISTORY_SECONDS = 30

# Prolonged closure threshold
PROLONGED_CLOSURE_SECONDS = 1.0

# Score limits
MAX_SCORE = 100.0

# Score added for a prolonged closure
PROLONGED_CLOSURE_POINTS = 25.0

# Score decay per second
SCORE_DECAY_PER_SECOND = 2.0


# ------------------------------------------------------------
# STATE
# ------------------------------------------------------------

eye_drowsiness_score = 0.0

closure_events = deque()

last_score_update = time.time()

eyes_closed = False

closure_start_time = None

current_closure_duration = 0.0

last_event = "None"

longest_closure = 0.0


print("=" * 60)
print("EYE DROWSINESS SCORE SYSTEM READY")
print("=" * 60)

print(
    f"Prolonged closure : "
    f"{PROLONGED_CLOSURE_SECONDS:.1f} sec"
)

print(
    f"Points per event  : "
    f"{PROLONGED_CLOSURE_POINTS:.1f}"
)

print(
    f"Score decay       : "
    f"{SCORE_DECAY_PER_SECOND:.1f}/sec"
)

print("=" * 60)

EYE DROWSINESS SCORE SYSTEM READY
Prolonged closure : 1.0 sec
Points per event  : 25.0
Score decay       : 2.0/sec


In [16]:
# ============================================================
# NB4 — BLOCK 13
# LIVE EYE DROWSINESS SCORE
# ============================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("=" * 60)
print("LIVE EYE DROWSINESS ANALYSIS")
print("=" * 60)
print("Blink normally.")
print("Try a prolonged eye closure.")
print("Press Q to quit.")
print("=" * 60)


# ------------------------------------------------------------
# Reset scoring state
# ------------------------------------------------------------

eye_drowsiness_score = 0.0

closure_events.clear()

last_score_update = time.time()

eyes_closed = False

closure_start_time = None

current_closure_duration = 0.0

last_event = "None"

longest_closure = 0.0


while True:

    ret, frame = cap.read()

    if not ret:
        print("Failed to read webcam frame.")
        break

    # --------------------------------------------------------
    # Mirror camera
    # --------------------------------------------------------

    frame = cv2.flip(frame, 1)

    h, w = frame.shape[:2]

    # --------------------------------------------------------
    # MediaPipe
    # --------------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = detector.detect(mp_image)

    # --------------------------------------------------------
    # Score decay
    # --------------------------------------------------------

    now = time.time()

    elapsed = now - last_score_update

    eye_drowsiness_score -= (
        SCORE_DECAY_PER_SECOND * elapsed
    )

    eye_drowsiness_score = max(
        0.0,
        eye_drowsiness_score
    )

    last_score_update = now


    if result.face_landmarks:

        landmarks = result.face_landmarks[0]

        # ----------------------------------------------------
        # EAR
        # ----------------------------------------------------

        left_ear = calculate_ear(
            landmarks,
            LEFT_EYE,
            w,
            h
        )

        right_ear = calculate_ear(
            landmarks,
            RIGHT_EYE,
            w,
            h
        )

        average_ear = (
            left_ear + right_ear
        ) / 2.0

        # ----------------------------------------------------
        # Eye states
        # ----------------------------------------------------

        left_state = classify_eye_state(
            left_ear
        )

        right_state = classify_eye_state(
            right_ear
        )

        both_closed = (
            left_state == "CLOSED"
            and
            right_state == "CLOSED"
        )


        # ====================================================
        # EYES CLOSED
        # ====================================================

        if both_closed:

            if not eyes_closed:

                eyes_closed = True

                closure_start_time = time.time()

                current_closure_duration = 0.0


            current_closure_duration = (
                time.time()
                - closure_start_time
            )


        # ====================================================
        # EYES OPEN
        # ====================================================

        else:

            if eyes_closed:

                # --------------------------------------------
                # Completed closure
                # --------------------------------------------

                duration = current_closure_duration

                event = classify_eye_closure(
                    duration
                )

                last_event = event


                # --------------------------------------------
                # Longest closure
                # --------------------------------------------

                if duration > longest_closure:

                    longest_closure = duration


                # --------------------------------------------
                # Prolonged closure
                # --------------------------------------------

                if (
                    duration
                    >= PROLONGED_CLOSURE_SECONDS
                ):

                    # Add event timestamp
                    closure_events.append(
                        time.time()
                    )

                    # Add drowsiness points
                    eye_drowsiness_score += (
                        PROLONGED_CLOSURE_POINTS
                    )


                # --------------------------------------------
                # Reset
                # --------------------------------------------

                eyes_closed = False

                closure_start_time = None

                current_closure_duration = 0.0


        # ====================================================
        # REMOVE OLD EVENTS
        # ====================================================

        cutoff_time = (
            time.time()
            - EVENT_HISTORY_SECONDS
        )

        while (
            closure_events
            and
            closure_events[0] < cutoff_time
        ):

            closure_events.popleft()


        # ----------------------------------------------------
        # Limit score
        # ----------------------------------------------------

        eye_drowsiness_score = min(
            MAX_SCORE,
            max(
                0.0,
                eye_drowsiness_score
            )
        )


        # ====================================================
        # STATUS
        # ====================================================

        if eye_drowsiness_score >= 70:

            status = "HIGH"

        elif eye_drowsiness_score >= 40:

            status = "MODERATE"

        else:

            status = "LOW"


        # ====================================================
        # DISPLAY
        # ====================================================

        cv2.putText(
            frame,
            f"Left EAR  : {left_ear:.3f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Right EAR : {right_ear:.3f}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Average EAR: {average_ear:.3f}",
            (20, 105),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Eyes: {'CLOSED' if both_closed else 'OPEN'}",
            (20, 145),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Closure: {current_closure_duration:.2f}s",
            (20, 185),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Last Event: {last_event}",
            (20, 225),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Recent Prolonged Closures: {len(closure_events)}",
            (20, 265),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Eye Drowsiness Score: {eye_drowsiness_score:.1f}/100",
            (20, 305),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 255),
            2
        )

        cv2.putText(
            frame,
            f"Eye Status: {status}",
            (20, 345),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.80,
            (0, 0, 255)
            if status == "HIGH"
            else (0, 255, 0),
            2
        )


    else:

        cv2.putText(
            frame,
            "FACE NOT DETECTED",
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 0, 255),
            2
        )


    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    cv2.imshow(
        "NB4 - Eye Drowsiness Analysis",
        frame
    )


    # --------------------------------------------------------
    # Quit
    # --------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


cap.release()

cv2.destroyAllWindows()

print("=" * 60)
print("EYE DROWSINESS ANALYSIS STOPPED")
print("=" * 60)

print(
    f"Final Eye Drowsiness Score: "
    f"{eye_drowsiness_score:.1f}/100"
)

print(
    f"Longest Closure: "
    f"{longest_closure:.2f} sec"
)

print("=" * 60)

LIVE EYE DROWSINESS ANALYSIS
Blink normally.
Try a prolonged eye closure.
Press Q to quit.
EYE DROWSINESS ANALYSIS STOPPED
Final Eye Drowsiness Score: 0.0/100
Longest Closure: 3.54 sec
